# Scheduling

Solvers in the **interval-schedule** shape: place work on a timeline.

In [ ]:
import pandas as pd
import plotly.graph_objects as go

import ortidy

## Shift scheduling

Staff each (day, shift) to its requirement, at most one shift per worker per day, balancing the load.

In [ ]:
requirements = pd.DataFrame({
    "day": [0, 0, 1, 1, 2, 2],
    "shift": ["am", "pm", "am", "pm", "am", "pm"],
    "required": [1, 1, 1, 1, 1, 1],
})
workers = pd.DataFrame({"workerId": ["alice", "bob", "carol"]})
result = ortidy.shift_scheduling(requirements, workers)
print(result.status, "| peak load:", result.objective)
result.frame.sort_values(["day", "shift"])

## Job shop

Schedule each job's task sequence on shared machines to minimize the makespan, then draw the Gantt chart.

In [ ]:
tasks = pd.DataFrame({
    "jobId": [0, 0, 0, 1, 1, 1, 2, 2, 2],
    "step": [0, 1, 2, 0, 1, 2, 0, 1, 2],
    "machine": ["m0", "m1", "m2", "m0", "m2", "m1", "m1", "m0", "m2"],
    "duration": [3, 2, 2, 2, 1, 4, 4, 3, 1],
})
result = ortidy.job_shop(tasks)
print(result.status, "| makespan:", result.objective)

sched = result.frame
fig = go.Figure()
for job, g in sched.groupby("jobId"):
    fig.add_bar(
        y=g["machine"], x=g["duration"], base=g["start"],
        orientation="h", name=f"job {job}",
        hovertext=[f"job {job} step {s}" for s in g["step"]],
    )
fig.update_layout(barmode="overlay", title="Job-shop schedule",
                  xaxis_title="time", yaxis_title="machine", height=400)
fig